# 02_Transformacion_Silver

Transformación de las cuatro tablas Bronze hacia Silver.

Objetivos de esta capa:

- eliminar columnas técnicas del CSV que no aportan al modelo (`_c0`);
- tipar fechas y campos numéricos;
- convertir coordenadas con coma decimal a `DOUBLE`;
- limpiar cadenas vacías y espacios;
- estandarizar algunas categorías sin alterar el grano natural;
- conservar metadatos de trazabilidad;
- validar conteos, claves e integridad referencial.

Las transformaciones se basan en la exploración previa de las tablas Bronze.


## 1. Configuración


In [ ]:
from pathlib import Path
import sys

# Ejecutar desde el Git Folder completo, no desde un notebook importado aislado.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "accidentalidad" / "policies.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("No se encontró el paquete accidentalidad; abra el repositorio completo en un Git Folder")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from accidentalidad.policies import sql_identifier, parse_bool, nonnegative_int

import uuid
from pyspark.sql import functions as F
from accidentalidad.quality import QualityGate

dbutils.widgets.text("catalog_name", "accidentalidad_databricks", "Catálogo")
dbutils.widgets.text("max_vehicle_orphans", "3", "Tolerancia de claves huérfanas en vehículos")
PROY_CATALOG = sql_identifier(dbutils.widgets.get("catalog_name"))
PROY_SCHEMA_BRONZE = "bronze"
PROY_SCHEMA_SILVER = "silver"
MAX_VEHICLE_ORPHANS = nonnegative_int(dbutils.widgets.get("max_vehicle_orphans"))
RUN_ID = uuid.uuid4().hex
BRONZE_VERSIONS = {}

def leer_bronze(nombre):
    tabla = f"{PROY_CATALOG}.bronze.{nombre}"
    version = spark.sql(f"DESCRIBE HISTORY {tabla}").select("version").first()["version"]
    BRONZE_VERSIONS[nombre] = version
    return (spark.read.option("versionAsOf", version).table(tabla)
            .withColumn("_bronze_version", F.lit(version))
            .withColumn("_quality_run_id", F.lit(RUN_ID)))

## 2. Funciones auxiliares

Se usan funciones tolerantes para que un valor inesperado no detenga todo el pipeline.

- `limpiar_texto`: elimina espacios y convierte cadenas vacías en `NULL`.
- `decimal_con_coma`: transforma valores como `-74,842098` en `-74.842098`.
- `fecha_ddmmyyyy`: convierte fechas de accidente en formato `dd/MM/yyyy`.
- `fecha_iso`: convierte fechas en formato `yyyy-MM-dd`.
- `fecha_nacimiento`: soporta tanto `yyyy-MM-dd` como `yyyy-MM-dd HH:mm:ss`.


In [ ]:
def limpiar_texto(columna):
    valor = F.trim(F.col(columna))
    return F.when(valor == "", F.lit(None)).otherwise(valor)


def decimal_con_coma(columna):
    return F.expr(
        f"try_cast(replace(trim(`{columna}`), ',', '.') AS DOUBLE)"
    )


def fecha_ddmmyyyy(columna):
    return F.expr(
        f"CAST(try_to_timestamp(trim(`{columna}`), 'dd/MM/yyyy') AS DATE)"
    )


def fecha_iso(columna):
    return F.expr(
        f"CAST(try_to_timestamp(trim(`{columna}`), 'yyyy-MM-dd') AS DATE)"
    )


def fecha_nacimiento(columna):
    return F.coalesce(
        F.to_date(
            F.expr(
                f"try_to_timestamp(`{columna}`, 'yyyy-MM-dd HH:mm:ss')"
            )
        ),
        F.expr(
            f"CAST(try_to_timestamp(trim(`{columna}`), 'yyyy-MM-dd') AS DATE)"
        ),
        F.expr(
            f"CAST(try_to_timestamp(trim(`{columna}`), 'dd/MM/yyyy') AS DATE)"
        )
    )


def entero_exacto(columna):
    # Rechaza fracciones, negativos fuera de dominio y overflow mediante controles posteriores.
    return F.expr(f"try_cast(try_cast(`{columna}` AS DECIMAL(38, 10)) AS INT)")

## 3. Transformación de accidentes

Grano: **una fila por accidente (`croquis`)**.

Decisiones principales:

- `_c0` se elimina porque corresponde al índice exportado del CSV.
- `fecha_accidente` pasa a `DATE`.
- año, mes y día pasan a `INT`.
- heridos y muertos pasan a `INT`, conservando `NULL` cuando la fuente no reporta valor.
- coordenadas pasan a `DOUBLE`.
- los campos categóricos principales se estandarizan en mayúsculas.


In [ ]:
df_accidentes_bronze = leer_bronze("accidentes")

df_accidentes_silver = (
    df_accidentes_bronze
    .drop("_c0")
    .withColumn("fecha_accidente", fecha_ddmmyyyy("fecha_accidente"))
    .withColumn("anio_accidente", F.expr("try_cast(anio_accidente AS INT)"))
    .withColumn("mes_accidente", F.expr("try_cast(mes_accidente AS INT)"))
    .withColumn("dia_accidente", F.expr("try_cast(dia_accidente AS INT)"))
    .withColumn("nro_heridos", entero_exacto("nro_heridos"))
    .withColumn("numero_muertos", entero_exacto("numero_muertos"))
    .withColumn("coordenada_x", decimal_con_coma("coordenada_x"))
    .withColumn("coordenada_y", decimal_con_coma("coordenada_y"))
    .withColumn("croquis", limpiar_texto("croquis"))
    .withColumn("hora_accidente", limpiar_texto("hora_accidente"))
    .withColumn("gravedad", F.upper(limpiar_texto("gravedad")))
    .withColumn("clase_accidente", F.upper(limpiar_texto("clase_accidente")))
    .withColumn("localidad", F.upper(limpiar_texto("localidad")))
    .withColumn("sitio_exacto", limpiar_texto("sitio_exacto"))
    .withColumn("placa_agente", F.upper(limpiar_texto("placa_agente")))
    .withColumn("apellidos", limpiar_texto("apellidos"))
    .withColumn("nombres", limpiar_texto("nombres"))
    .withColumn("_silver_transformation_timestamp", F.current_timestamp())
)

display(df_accidentes_silver.limit(10))
df_accidentes_silver.printSchema()

## 4. Transformación de causas

Grano: **una fila por accidente (`croquis`)**.

Además del tipado de la fecha, se corrige el carácter `¤` observado en textos como `Desobedecer se¤ales`, interpretándolo como `ñ`.


In [ ]:
df_causas_bronze = leer_bronze("causas")

df_causas_silver = (
    df_causas_bronze
    .drop("_c0")
    .withColumn("fecha_accidente", fecha_ddmmyyyy("fecha_accidente"))
    .withColumn("croquis", limpiar_texto("croquis"))
    .withColumn(
        "hipotesis_causa",
        F.regexp_replace(limpiar_texto("hipotesis_causa"), "¤", "ñ")
    )
    .withColumn("_silver_transformation_timestamp", F.current_timestamp())
)

display(df_causas_silver.limit(10))
df_causas_silver.printSchema()

## 5. Transformación de vehículos

Grano: **una fila por vehículo involucrado**.

La exploración mostró dos formatos de fecha diferentes:

- `fecha_accidente`: `dd/MM/yyyy`
- `fecha_radica`: `yyyy-MM-dd`

Por ello cada campo se convierte con su formato correspondiente.


In [ ]:
df_vehiculos_bronze = leer_bronze("vehiculos")

df_vehiculos_silver = (
    df_vehiculos_bronze
    .drop("_c0")
    .withColumn("fecha_accidente", fecha_ddmmyyyy("fecha_accidente"))
    .withColumn("fecha_radica", fecha_iso("fecha_radica"))
    .withColumn("anio_accidente", F.expr("try_cast(anio_accidente AS INT)"))
    .withColumn("mes_accidente", F.expr("try_cast(mes_accidente AS INT)"))
    .withColumn("dia_accidente", F.expr("try_cast(dia_accidente AS INT)"))
    .withColumn("coordenada_x", decimal_con_coma("coordenada_x"))
    .withColumn("coordenada_y", decimal_con_coma("coordenada_y"))
    .withColumn("croquis", limpiar_texto("croquis"))
    .withColumn("hora_accidente", limpiar_texto("hora_accidente"))
    .withColumn("direccion", limpiar_texto("direccion"))
    .withColumn("placa", F.upper(limpiar_texto("placa")))
    .withColumn("clase", F.upper(limpiar_texto("clase")))
    .withColumn("servicio", F.upper(limpiar_texto("servicio")))
    .withColumn("clase_accidente", F.upper(limpiar_texto("clase_accidente")))
    .withColumn("_silver_transformation_timestamp", F.current_timestamp())
)

display(df_vehiculos_silver.limit(10))
df_vehiculos_silver.printSchema()

## 6. Transformación de víctimas

Grano: **una fila por víctima**.

La exploración permitió identificar que:

- `clase` contiene la clase del accidente (`Choque`, `Atropello`, etc.);
- `clase_1` contiene la clase del vehículo (`MOTOCICLETA`, `AUTOMOVIL`, etc.).

Por claridad semántica se renombran a `clase_accidente` y `clase_vehiculo`.

`fecha_nacimiento` aparece en formatos tipo `yyyy-MM-dd HH:mm:ss` y `yyyy-MM-dd`, por lo que se convierte de manera tolerante.


In [ ]:
df_victimas_bronze = leer_bronze("victimas")

df_victimas_silver = (
    df_victimas_bronze
    .drop("_c0")
    .withColumnRenamed("clase", "clase_accidente")
    .withColumnRenamed("clase_1", "clase_vehiculo")
    .withColumn("fecha_accidente", fecha_ddmmyyyy("fecha_accidente"))
    .withColumn("fecha_nacimiento", fecha_nacimiento("fecha_nacimiento"))
    .withColumn("anio_accidente", F.expr("try_cast(anio_accidente AS INT)"))
    .withColumn("mes_accidente", F.expr("try_cast(mes_accidente AS INT)"))
    .withColumn("dia_accidente", F.expr("try_cast(dia_accidente AS INT)"))
    .withColumn("edad", F.expr("try_cast(edad AS INT)"))
    .withColumn("coordenada_x", decimal_con_coma("coordenada_x"))
    .withColumn("coordenada_y", decimal_con_coma("coordenada_y"))
    .withColumn("croquis", limpiar_texto("croquis"))
    .withColumn("hora_accidente", limpiar_texto("hora_accidente"))
    .withColumn("direccion", limpiar_texto("direccion"))
    .withColumn("descripcion_victima", F.upper(limpiar_texto("descripcion_victima")))
    .withColumn("gravedad", F.upper(limpiar_texto("gravedad")))
    .withColumn("id_usuario", limpiar_texto("id_usuario"))
    .withColumn("sexo", F.upper(limpiar_texto("sexo")))
    .withColumn("placa", F.upper(limpiar_texto("placa")))
    .withColumn("clase_accidente", F.upper(limpiar_texto("clase_accidente")))
    .withColumn("clase_vehiculo", F.upper(limpiar_texto("clase_vehiculo")))
    .withColumn("servicio", F.upper(limpiar_texto("servicio")))
    .withColumn("_silver_transformation_timestamp", F.current_timestamp())
)

display(df_victimas_silver.limit(10))
df_victimas_silver.printSchema()

## 7. Preparación de las validaciones
Todavía no se publica Silver. Los controles se realizan sobre los DataFrames candidatos,
leídos desde versiones fijas de Bronze. Los fallos críticos bloquean todas las escrituras Silver.

In [ ]:
tablas_silver = {
    "accidentes": df_accidentes_silver, "causas": df_causas_silver,
    "vehiculos": df_vehiculos_silver, "victimas": df_victimas_silver,
}
tablas_bronze = {
    "accidentes": df_accidentes_bronze, "causas": df_causas_bronze,
    "vehiculos": df_vehiculos_bronze, "victimas": df_victimas_bronze,
}
gate = QualityGate(spark, PROY_CATALOG, RUN_ID, "silver")

## 8. Validación de conteos Bronze vs Silver

Las transformaciones Silver no deben cambiar el número de filas de ninguna fuente.


In [ ]:
for nombre, df in tablas_silver.items():
    total = df.count()
    gate.check(nombre, "row_count", total, tablas_bronze[nombre].count())
    gate.check(nombre, "empty_dataset", total == 0, False)

## 9. Validación de claves y granularidad

Se revisa el número de accidentes distintos y la ausencia de claves nulas.

Esperado:

- accidentes: 23,212 `croquis` distintos;
- causas: 23,212 `croquis` distintos;
- vehículos: 23,215 `croquis` distintos;
- víctimas: 7,758 `croquis` distintos.


In [ ]:
for nombre, df in tablas_silver.items():
    gate.check(nombre, "null_croquis", df.filter(F.col("croquis").isNull()).count())
    if nombre in {"accidentes", "causas"}:
        duplicados = df.groupBy("croquis").count().filter(F.col("count") > 1).count()
        gate.check(nombre, "duplicate_croquis", duplicados)

## 10. Validación de conversiones críticas

Si una columna fuente era no nula y después del tipado queda `NULL`, significa que algún valor no pudo convertirse y debe investigarse.


In [ ]:
# Detectar toda pérdida de valores al convertir, también en campos de calendario.
conversiones = {
    "accidentes": ["fecha_accidente", "anio_accidente", "mes_accidente", "dia_accidente", "nro_heridos", "numero_muertos", "coordenada_x", "coordenada_y"],
    "causas": ["fecha_accidente"],
    "vehiculos": ["fecha_accidente", "fecha_radica", "anio_accidente", "mes_accidente", "dia_accidente", "coordenada_x", "coordenada_y"],
    "victimas": ["fecha_accidente", "fecha_nacimiento", "anio_accidente", "mes_accidente", "dia_accidente", "edad", "coordenada_x", "coordenada_y"],
}
for nombre, columnas in conversiones.items():
    raw, clean = tablas_bronze[nombre], tablas_silver[nombre]
    for columna in columnas:
        raw_nulls = raw.filter(F.col(columna).isNull() | (F.trim(F.col(columna)) == "")).count()
        clean_nulls = clean.filter(F.col(columna).isNull()).count()
        gate.check(nombre, f"conversion_{columna}", clean_nulls - raw_nulls)
    for campo, parte in [("anio_accidente", F.year("fecha_accidente")), ("mes_accidente", F.month("fecha_accidente")), ("dia_accidente", F.dayofmonth("fecha_accidente"))]:
        if campo in clean.columns:
            inconsistentes = clean.filter(~F.col(campo).eqNullSafe(parte)).count()
            gate.check(nombre, f"calendar_{campo}", inconsistentes)
    for campo in ["nro_heridos", "numero_muertos", "edad"]:
        if campo in clean.columns:
            limites = (F.col(campo) < 0) | ((F.col(campo) > 120) if campo == "edad" else F.lit(False))
            gate.check(nombre, f"range_{campo}", clean.filter(limites).count())
            # La conversión a entero no puede truncar silenciosamente 1.5 a 1.
            valor = F.expr(f"try_cast(`{campo}` AS DOUBLE)")
            fracciones = raw.filter(valor.isNotNull() & ((valor != F.floor(valor)) | F.isnan(valor))).count()
            gate.check(nombre, f"fractional_{campo}", fracciones)
    for campo in ["coordenada_x", "coordenada_y"]:
        if campo in clean.columns:
            invalidas = clean.filter(F.isnan(campo) | (F.abs(F.col(campo)) > 180)).count()
            gate.check(nombre, f"finite_geographic_{campo}", invalidas)

## 11. Integridad referencial
Causas y víctimas no admiten huérfanos. Vehículos admite hasta `max_vehicle_orphans`
claves distintas (3 para el lote histórico documentado; use 0 para modo estricto).
Se guardan las filas afectadas en `quality.orphan_<dataset>` y se registra una advertencia.
Esta tolerancia es cuantitativa y no garantiza que sean las mismas claves del lote histórico.

In [ ]:
accidentes_keys = df_accidentes_silver.select("croquis").distinct()
for nombre in ["causas", "vehiculos", "victimas"]:
    df = tablas_silver[nombre]
    keys = df.select("croquis").distinct().join(accidentes_keys, "croquis", "left_anti")
    total = keys.count()
    if total:
        # Conservar las filas completas y el identificador de la ejecución, sin eliminarlas de Silver.
        (df.join(keys, "croquis", "inner").withColumn("_dataset", F.lit(nombre))
         .write.format("delta").mode("append").option("mergeSchema", "true")
         .saveAsTable(f"{PROY_CATALOG}.quality.orphan_{nombre}"))
    limite = MAX_VEHICLE_ORPHANS if nombre == "vehiculos" else 0
    gate.check(nombre, "orphan_limit", total <= limite, True, details={"count": total, "limit": limite})
    if total and total <= limite:
        gate.check(nombre, "accepted_orphans", total, severity="WARN", details={"limit": limite})

## 12. Consistencia entre accidentes y víctimas

Se valida que la cantidad de víctimas registrada por accidente sea consistente con la suma de heridos y muertos reportada en `silver.accidentes`.

Para esta validación, los `NULL` de heridos o muertos se interpretan como cero **solo dentro del cálculo de control**, sin modificar los valores almacenados en Silver.


In [ ]:
victimas_por_accidente = df_victimas_silver.groupBy("croquis").agg(F.count("*").alias("cantidad_victimas"))
control = (
    df_accidentes_silver.select("croquis", (F.coalesce(F.col("nro_heridos"), F.lit(0)) + F.coalesce(F.col("numero_muertos"), F.lit(0))).alias("reportadas"))
    .join(victimas_por_accidente, "croquis", "left").fillna({"cantidad_victimas": 0})
)
gate.check("accidentes", "victim_reconciliation", control.filter(F.col("reportadas") != F.col("cantidad_victimas")).count())
gate.finish()

for nombre, df in tablas_silver.items():
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable(f"{PROY_CATALOG}.silver.{nombre}"))
    print(f"[OK] Publicado Silver: {nombre}, Bronze v{BRONZE_VERSIONS[nombre]}, run_id={RUN_ID}")

## 13. Resultado

Al finalizar correctamente este notebook deben existir:

- `silver.accidentes`
- `silver.causas`
- `silver.vehiculos`
- `silver.victimas`

Estas tablas serán la base para las siguientes etapas del proyecto: Delta Lake, controles adicionales, modelo dimensional Gold y dataset de Machine Learning.
